# 01. Aquisição, qualidade e EDA do subconjunto Disque 100

**Escopo:** denúncias de tráfico de pessoas e violações coocorrentes.

**Unidade física do arquivo:** linha do CSV. A unidade substantiva será validada por `hash`, pois uma denúncia pode ocupar várias linhas, especialmente quando há mais de uma `violacao`.

**Saídas:** dados limpos em nível de linha, base agregada por denúncia, matriz binária de violações, relatório de correções e comparação antes/depois.


## 1. Configuração do ambiente


In [1]:
from pathlib import Path
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

DATA_FILE = Path("data/trafico-pessoas-com-outras-violacoes-2021-2025.csv")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIMIAR_AUSENCIA_COLUNA = 0.80
RANDOM_STATE = 42


## 2. Carregamento e validação do esquema


In [3]:
COLUNAS_ESPERADAS = ['hash', 'data_de_cadastro', 'canal_de_atendimento', 'denuncia_emergencial', 'denunciante', 'cenario_da_violacao', 'pais', 'uf', 'municipio', 'frequencia', 'inicio_das_violacoes', 'sl_quantidade_vitimas', 'grupo_vulneravel', 'motivacao', 'relacao_vitima_suspeito', 'sl_vitima_cadastro', 'genero_da_vitima', 'orientacao_sexual_da_vitima', 'faixa_etaria_da_vitima', 'nacionalidade_da_vitima', 'sl_vitima_naturalidade', 'sl_vitima_naturalizado_uf', 'sl_vitima_naturalizado_municipio', 'deficiencia_da_vitima', 'doenca_rara_da_vitima', 'deficiencia_relacionada_a_doenca_rara', 'vitima_preso_a', 'pais_da_vitima', 'uf_da_vitima', 'municipio_da_vitima', 'profissao_da_vitima', 'grau_de_instrucao_da_vitima', 'religiao_da_vitima', 'raca_cor_da_vitima', 'etnia_da_vitima', 'faixa_de_renda_da_vitima', 'natureza_juridica_do_suspeito', 'genero_do_suspeito', 'orientacao_sexual_do_suspeito', 'faixa_etaria_do_suspeito', 'suspeito_nacionalidade', 'sl_suspeito_naturalidade', 'sl_suspeito_naturalizado_uf', 'sl_suspeito_naturalizado_municipio', 'deficiencia_do_suspeito', 'doenca_rara_do_suspeito', 'deficiencia_relacionada_a_doenca_rara_suspeito', 'suspeito_preso', 'pais_do_suspeito', 'uf_do_suspeito', 'sl_suspeito_municipio', 'profissao_do_suspeito', 'grau_de_instrucao_do_suspeito', 'religiao_do_suspeito', 'raca_cor_do_suspeito', 'suspeito_etnia', 'faixa_de_renda_do_suspeito', 'vinculo_orgao_pj_do_suspeito', 'sl_suspeito_ramo', 'suspeito_especificacao_etnia', 'vitima_especificacao_etnia', 'violacao']

df_raw = pd.read_csv(DATA_FILE, sep=";", encoding="utf-8", low_memory=False)
faltantes = sorted(set(COLUNAS_ESPERADAS) - set(df_raw.columns))
extras = sorted(set(df_raw.columns) - set(COLUNAS_ESPERADAS))
assert not faltantes, f"Colunas esperadas ausentes: {faltantes}"
print("Colunas extras:", extras)
print("Dimensão bruta:", df_raw.shape)
df_raw.head()


Colunas extras: []
Dimensão bruta: (8401, 62)


,hash,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,vinculo_orgao_pj_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
0,79c48e0d422d3ab1a306f8d7e498b8be,2021-01-08 23:11:00,TELEFÔNICO,NÃO,ANÔNIMO,SERVIÇO DE ABRIGAMENTO,BRASIL,SP,SÃO BERNARDO DO CAMPO,DIARIAMENTE,HÁ MAIS DE CINCO ANOS,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,PARA OBTENÇÃO DE BENEFÍCIO FINANCEIRO/ GANÂNCIA,OUTROS,PESSOA FÍSICA,FEMININO,N/D,NaN,N/D,NaN,NaN,NaN,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,60 A 64 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>CALÚNIA
1,0b9e67b28384351a4006c49ba3ff66b7,2021-01-14 07:11:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO A...,FAMILIARES,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,ATÉ 1 SM,PESSOA FÍSICA,FEMININO,N/D,35 A 39 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO PAULO,N/D,ENSINO FUNDAMENTAL INCOMPLETO,N/D,BRANCA,N/D,ATÉ 1 SM,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>AMEAÇA ou COAÇÃO
2,6fa1ed73ec4c3f385683c9fc03a3c062,2021-01-12 07:34:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA DA VÍTIMA,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,20,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO A...,PAI,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,45 A 49 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO SABE INFORMAR,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,PRETA,N/D,N/D,NaN,N/D,N/D,N/D,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESS...
3,79c48e0d422d3ab1a306f8d7e498b8be,2021-01-08 23:11:00,TELEFÔNICO,NÃO,ANÔNIMO,SERVIÇO DE ABRIGAMENTO,BRASIL,SP,SÃO BERNARDO DO CAMPO,DIARIAMENTE,HÁ MAIS DE CINCO ANOS,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,RESULTANDO EM LESÃO GRAVÍSSIMA,OUTROS,PESSOA FÍSICA,FEMININO,N/D,NaN,N/D,NaN,NaN,NaN,N/D,NÃO TEM DOENÇA RARA,NaN,N/D,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,60 A 64 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,NaN,N/D,N/D,N/D,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESS...
4,ba12fedf0ff1a3bbe12b853e0ae2da65,2021-01-13 02:28:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BRASIL,SP,SÃO PAULO,DIARIAMENTE,NÃO SABE INFORMAR,10,VIOLÊNCIA CONTRA PESSOA SOCIALMENTE VULNERÁVEL,PARA 

### 2.1 Conversão dos tipos conhecidos


In [ ]:
df = df_raw.copy(deep=True)
df["data_de_cadastro"] = pd.to_datetime(df["data_de_cadastro"], format="mixed")
df["data_de_cadastro"] = df["data_de_cadastro"].dt.floor("s") 

df["sl_quantidade_vitimas"] = pd.to_numeric(df["sl_quantidade_vitimas"], errors="coerce").astype("Int64")

erros_data = int(df["data_de_cadastro"].isna().sum() - df_raw["data_de_cadastro"].isna().sum())
erros_quantidade = int(df["sl_quantidade_vitimas"].isna().sum() - df_raw["sl_quantidade_vitimas"].isna().sum())
print({"datas_nao_convertidas": erros_data, "quantidades_nao_convertidas": erros_quantidade})
df.info()


{'datas_nao_convertidas': 0, 'quantidades_nao_convertidas': 0}
<class 'pandas.DataFrame'>
RangeIndex: 8401 entries, 0 to 8400
Data columns (total 62 columns):
 #   Column                                          Non-Null Count  Dtype         
---  ------                                          --------------  -----         
 0   hash                                            8401 non-null   str           
 1   data_de_cadastro                                8401 non-null   datetime64[us]
 2   canal_de_atendimento                            8401 non-null   str           
 3   denuncia_emergencial                            8401 non-null   str           
 4   denunciante                                     8401 non-null   str           
 5   cenario_da_violacao                             8401 non-null   str           
 6   pais                                            8401 non-null   str           
 7   uf                                              7828 non-null   str           
 

## 3. Compreensão da granularidade


In [ ]:
granularidade = pd.Series({
    "linhas": len(df),
    "hash_unicos": df["hash"].nunique(dropna=True),
    "hash_repetidos_linhas": int(df["hash"].duplicated(keep=False).sum()),
    "linhas_duplicadas_exatas": int(df.duplicated().sum()),
    "violacoes_unicas": df["violacao"].nunique(dropna=True)
})
display(granularidade.to_frame("valor"))

linhas_por_denuncia = df.groupby("hash", dropna=False).size()
display(linhas_por_denuncia.describe().to_frame("linhas_por_hash"))


### Decisão metodológica

- Preserve a base em nível de linha para auditoria e EDA.
- Para clusterização e detecção de anomalias, agregue por `hash` para evitar que denúncias com várias violações recebam peso maior.
- Para associação, use uma matriz denúncia × violação, também indexada por `hash`.


## 4. Avaliação da qualidade dos dados


### 4.1 Completude explícita


In [ ]:
relatorio_missing_explicito = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "quantidade": df.isna().sum(),
    "percentual": df.isna().mean().mul(100).round(2),
    "unicos": df.nunique(dropna=True)
}).sort_values("percentual", ascending=False)
relatorio_missing_explicito


### 4.2 Valores ausentes implícitos


In [ ]:
MISSING_TEXT = {
    "", "NA", "N/A", "NULL", "NULO", "IGNORADO", "NAO INFORMADO",
    "NAO SABE", "NAO SE APLICA", "SEM INFORMACAO", "SEM PREENCHIMENTO",
    "DESCONHECIDO", "INDEFINIDO", "OUTROS/NAO INFORMADO"
}
MISSING_CODES = {-95, -97, -98, -99, -999}

def normalizar_texto(v):
    if pd.isna(v): return np.nan
    t = re.sub(r"\s+", " ", str(v)).strip().upper()
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))

def mascara_missing_implicito(s):
    if pd.api.types.is_numeric_dtype(s): return s.isin(MISSING_CODES)
    n = s.map(normalizar_texto)
    return n.isin(MISSING_TEXT | {str(x) for x in MISSING_CODES})

implicit_rows=[]
for col in df.columns:
    m=mascara_missing_implicito(df[col])
    implicit_rows.append({"coluna":col,"quantidade":int(m.sum()),"percentual":round(m.mean()*100,2)})
relatorio_missing_implicito=pd.DataFrame(implicit_rows)
relatorio_missing_implicito.query("quantidade > 0").sort_values("quantidade",ascending=False)


### 4.3 Consistência textual


In [ ]:
rows=[]
for col in df.select_dtypes(include=["object","string","category"]).columns:
    original=df[col].dropna().astype(str)
    norm=original.map(normalizar_texto)
    rows.append({"coluna":col,"categorias_originais":original.nunique(),"categorias_normalizadas":norm.nunique(),"reducao_potencial":original.nunique()-norm.nunique()})
relatorio_consistencia=pd.DataFrame(rows).sort_values("reducao_potencial",ascending=False)
relatorio_consistencia


### 4.4 Conformidade


In [ ]:
UFS = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}
COLUNAS_UF = ["uf","uf_da_vitima","sl_vitima_naturalizado_uf","uf_do_suspeito","sl_suspeito_naturalizado_uf"]
conformidade=[]
for col in COLUNAS_UF:
    valores=df[col].map(normalizar_texto)
    invalidos=valores.notna() & ~valores.isin(UFS)
    conformidade.append({"coluna":col,"invalidos":int(invalidos.sum()),"exemplos":sorted(valores[invalidos].unique())[:10]})

q=df["sl_quantidade_vitimas"]
conformidade.append({"coluna":"sl_quantidade_vitimas","invalidos":int((q.notna() & (q<1)).sum()),"exemplos":sorted(q[q.notna() & (q<1)].unique().tolist())})
pd.DataFrame(conformidade)


### 4.5 Unicidade


In [ ]:
print("Duplicatas exatas:", int(df.duplicated().sum()))
print("Duplicatas por hash + violacao:", int(df.duplicated(subset=["hash","violacao"]).sum()))
df[df.duplicated(subset=["hash","violacao"], keep=False)].sort_values(["hash","violacao"]).head(20)


### 4.6 Relatório de correções aplicadas


In [ ]:
correcoes=[]
def registrar(etapa,objeto,quantidade,regra,justificativa):
    correcoes.append({"etapa":etapa,"objeto":objeto,"quantidade":int(quantidade),"regra":regra,"justificativa":justificativa})

df_antes_limpeza=df.copy(deep=True)

# Ausências implícitas
for col in df.columns:
    m=mascara_missing_implicito(df[col]); n=int(m.sum())
    if n:
        df.loc[m,col]=np.nan
        registrar("Conversão de ausência implícita",col,n,"Marcadores semânticos/códigos -> NaN","Representação uniforme da ausência")

# Texto
for col in df.select_dtypes(include=["object","string","category"]).columns:
    antes=df[col].copy(); depois=df[col].map(normalizar_texto)
    n=int((antes.fillna("<NA>").astype(str)!=depois.fillna("<NA>").astype(str)).sum())
    df[col]=depois
    if n: registrar("Padronização textual",col,n,"Maiúsculas, trim, espaços e acentos","Uniformizar categorias")

# Duplicatas exatas
n=int(df.duplicated().sum())
if n:
    df=df.drop_duplicates().copy()
    registrar("Remoção de duplicatas","linhas",n,"Duplicatas exatas","Unicidade")

# Colunas >80% ausentes após normalização
pct=df.isna().mean()
cols_alta_ausencia=pct[pct>LIMIAR_AUSENCIA_COLUNA].index.tolist()
for col in cols_alta_ausencia:
    registrar("Remoção por alta ausência",col,int(df[col].isna().sum()),">80% ausente","Baixa disponibilidade informacional")
df=df.drop(columns=cols_alta_ausencia)

# Linhas totalmente vazias, desconsiderando hash se existir
cols_conteudo=[c for c in df.columns if c!="hash"]
m=df[cols_conteudo].isna().all(axis=1); n=int(m.sum())
if n:
    df=df.loc[~m].copy(); registrar("Remoção de linha vazia","linhas",n,"100% do conteúdo ausente","Sem informação analisável")

relatorio_correcoes=pd.DataFrame(correcoes)
relatorio_correcoes


**Observação esperada:** `vinculo_orgao_pj_do_suspeito` possui 100% de ausência no resumo fornecido e será removida. Também tendem a ultrapassar 80%: naturalidade/naturalização, especificações de etnia, algumas variáveis de deficiência, escolaridade, religião e renda. A decisão final é calculada após a conversão dos ausentes implícitos.


### 4.7 Comparação antes × depois


In [ ]:
def metricas(d):
    return {"linhas":len(d),"colunas":d.shape[1],"celulas":d.size,"ausencias_explicitas":int(d.isna().sum().sum()),"pct_ausente":round(d.isna().sum().sum()/d.size*100,2),"duplicatas_exatas":int(d.duplicated().sum()),"hash_unicos":d["hash"].nunique()}
comparacao=pd.DataFrame({"antes":metricas(df_antes_limpeza),"depois":metricas(df)})
comparacao["variacao_absoluta"]=comparacao["depois"]-comparacao["antes"]
comparacao


## 5. Engenharia inicial para EDA


In [ ]:
df["ano_cadastro"]=df["data_de_cadastro"].dt.year
df["mes_cadastro"]=df["data_de_cadastro"].dt.month
df["trimestre_cadastro"]=df["data_de_cadastro"].dt.quarter
df["dia_semana_cadastro"]=df["data_de_cadastro"].dt.dayofweek

MAPA_REGIAO={"AC":"NORTE","AP":"NORTE","AM":"NORTE","PA":"NORTE","RO":"NORTE","RR":"NORTE","TO":"NORTE","AL":"NORDESTE","BA":"NORDESTE","CE":"NORDESTE","MA":"NORDESTE","PB":"NORDESTE","PE":"NORDESTE","PI":"NORDESTE","RN":"NORDESTE","SE":"NORDESTE","DF":"CENTRO-OESTE","GO":"CENTRO-OESTE","MT":"CENTRO-OESTE","MS":"CENTRO-OESTE","ES":"SUDESTE","MG":"SUDESTE","RJ":"SUDESTE","SP":"SUDESTE","PR":"SUL","RS":"SUL","SC":"SUL"}
df["regiao_ocorrencia"]=df["uf"].map(MAPA_REGIAO)
df["regiao_vitima"]=df["uf_da_vitima"].map(MAPA_REGIAO)
df["regiao_suspeito"]=df["uf_do_suspeito"].map(MAPA_REGIAO)


## 6. EDA em nível de denúncia


In [ ]:
# Evita contagem duplicada quando um hash possui várias violações.
denuncias=df.sort_values("data_de_cadastro").drop_duplicates("hash",keep="first").copy()
print("Denúncias únicas:",len(denuncias))

def grafico_frequencia(col,n=15):
    s=denuncias[col].value_counts(dropna=False).head(n).sort_values()
    s.plot(kind="barh",figsize=(9,max(4,n*.3)),title=col)
    plt.xlabel("Denúncias únicas"); plt.tight_layout(); plt.show()

for col in ["canal_de_atendimento","denuncia_emergencial","cenario_da_violacao","frequencia","grupo_vulneravel","genero_da_vitima","faixa_etaria_da_vitima","raca_cor_da_vitima","genero_do_suspeito","faixa_etaria_do_suspeito","relacao_vitima_suspeito","regiao_ocorrencia"]:
    if col in denuncias.columns: grafico_frequencia(col)


### 6.1 Série temporal


In [ ]:
serie=denuncias.set_index("data_de_cadastro").resample("MS").size()
serie.plot(figsize=(12,5),title="Denúncias únicas por mês")
plt.ylabel("Quantidade"); plt.tight_layout(); plt.show()


### 6.2 Violações e coocorrências


In [ ]:
# Funciona tanto com uma violação por linha quanto com valores separados por ; ou |.
viol_long=(df[["hash","violacao"]].dropna().assign(violacao=lambda x:x["violacao"].str.split(r"\s*[;|]\s*",regex=True)).explode("violacao"))
viol_long["violacao"]=viol_long["violacao"].str.strip()
viol_long=viol_long[viol_long["violacao"].ne("")].drop_duplicates()
matriz_violacoes=pd.crosstab(viol_long["hash"],viol_long["violacao"]).clip(upper=1).astype("int8")

freq=matriz_violacoes.sum().sort_values(ascending=False)
display(freq.head(30).to_frame("denuncias"))
cooc=matriz_violacoes.T.dot(matriz_violacoes)
np.fill_diagonal(cooc.values,0)
display(cooc.stack().sort_values(ascending=False).head(30).to_frame("coocorrencias"))


## 7. Agregação por hash para modelagem


In [ ]:
def moda_segura(s):
    m=s.dropna().mode()
    return m.iloc[0] if not m.empty else np.nan

# Colunas descritivas: moda. Quantidade: máximo. Data: mínimo.
model_base=df.groupby("hash",as_index=True).agg({
    "data_de_cadastro":"min",
    "sl_quantidade_vitimas":"max",
    **{c:moda_segura for c in df.columns if c not in {"hash","data_de_cadastro","sl_quantidade_vitimas","violacao"}}
})
model_base=model_base.join(matriz_violacoes.add_prefix("violacao__"),how="left")
viol_cols=[c for c in model_base.columns if c.startswith("violacao__")]
model_base[viol_cols]=model_base[viol_cols].fillna(0).astype("int8")
model_base["qtd_tipos_violacao"]=model_base[viol_cols].sum(axis=1)
model_base["multiplas_violacoes"]=(model_base["qtd_tipos_violacao"]>1).astype("int8")
model_base.shape


## 8. Exportação


In [ ]:
df.to_parquet(OUTPUT_DIR/"disque100_linhas_tratadas.parquet",index=False)
model_base.reset_index().to_parquet(OUTPUT_DIR/"disque100_denuncias_modelagem.parquet",index=False)
matriz_violacoes.reset_index().to_parquet(OUTPUT_DIR/"matriz_violacoes_por_denuncia.parquet",index=False)
relatorio_correcoes.to_csv(OUTPUT_DIR/"relatorio_correcoes.csv",index=False,encoding="utf-8-sig")
comparacao.to_csv(OUTPUT_DIR/"comparacao_antes_depois.csv",encoding="utf-8-sig")
relatorio_missing_explicito.to_csv(OUTPUT_DIR/"relatorio_missing_explicito.csv",encoding="utf-8-sig")
print("Arquivos exportados para",OUTPUT_DIR.resolve())


## 9. Registro de decisões e achados

Documente versões, filtros, categorias consolidadas, colunas removidas e justificativas.
